# Setup

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/<name>/.local/bin:$PATH"

# Create a virtual environment
!/home/<name>/.local/bin/uv venv .venv --seed --clear

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [4]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "numpy<2" \
#     "torch==2.1.2+cu118" \
#     "transformers==4.51.3" \
#     "accelerate==0.34.2" \
#     "huggingface_hub>=0.23.0" \
#     "safetensors" \
#     "sentencepiece" \
#     "tqdm" \
#     "pandas" \
#     "matplotlib" \
#     "sympy" \
#     "antlr4-python3-runtime==4.11.1" \
#     --extra-index-url https://download.pytorch.org/whl/cu118

In [5]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "nvidia-cusparse-cu11" \
#     "nvidia-cublas-cu11" \
#     "nvidia-cuda-runtime-cu11" \
#     "nvidia-cudnn-cu11"

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "numpy<2" \
    "torch==2.3.1+cu121" \
    "torchvision==0.18.1+cu121" \
    "torchaudio==2.3.1+cu121" \
    --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "transformers==4.51.3" \
    "accelerate>=0.30.0" \
    "huggingface_hub>=0.23.0" \
    "safetensors" \
    "sentencepiece" \
    "tokenizers==0.21.4" \
    "sympy" \
    "pandas" \
    "matplotlib" \
    "tqdm" \
    "prettyprint" \
    "antlr4-python3-runtime==4.11.1" \
    "ipykernel" \
    "jupyter"

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "bitsandbytes==0.45.5"

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import os
import sys
import json
import time
import csv
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE1_DIR = RESULTS_DIR / "baseline1_weakest"

VAL_FRAC = 0.20
SPLIT_SEED = 414

CACHE_DIR = None
HF_HOME_DIR = None

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS_SMOKE = 512
MAX_NEW_TOKENS_BASELINE = 1024

BATCH_SIZE = 1
LOAD_IN_4BIT = True

BASELINE1_DIR.mkdir(parents=True, exist_ok=True)

MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

if HF_HOME_DIR is not None:
    os.environ["HF_HOME"] = str(HF_HOME_DIR)

if CACHE_DIR is not None:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("HF_HOME      :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE :", os.environ.get("HF_HUB_CACHE"))
print("cache_dir    :", CACHE_DIR)

HF_HOME      : None
HF_HUB_CACHE : None
cache_dir    : None


In [3]:
import torch

print(f"CUDA_VISIBLE_DEVICES (Env): {os.environ.get('CUDA_VISIBLE_DEVICES')}")

cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("PyTorch still can't see the GPU.")
    device = torch.device("cpu")

CUDA_VISIBLE_DEVICES (Env): 0
Is CUDA available? True
Current Device: 0
Device Name: NVIDIA GeForce GTX 1080 Ti


In [4]:
# import site

# roots = [Path(p) for p in site.getsitepackages()]
# matches = []

# for root in roots:
#     if root.exists():
#         matches.extend(root.rglob("libcusparse.so*"))

# for m in matches:
#     print(m)

In [5]:
# wanted_libs = {
#     "libcusparse.so",
#     "libcublas.so",
#     "libcudart.so",
#     "libcudnn.so",
# }

# lib_dirs = []

# for root in map(Path, site.getsitepackages()):
#     if not root.exists():
#         continue

#     for lib in wanted_libs:
#         for match in root.rglob(lib + "*"):
#             lib_dir = str(match.parent)
#             if lib_dir not in lib_dirs:
#                 lib_dirs.append(lib_dir)

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# cuda11_dirs = [str(p) for p in cuda11_dirs if p.exists()]

# path_line = ":".join(cuda11_dirs)

# print("Add this before starting the notebook/kernel:")
# print(f'export LD_LIBRARY_PATH="{path_line}:$LD_LIBRARY_PATH"')

In [6]:
# import os
# import subprocess
# from pathlib import Path

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# env = os.environ.copy()
# env["LD_LIBRARY_PATH"] = ":".join(str(p) for p in cuda11_dirs if p.exists()) + ":" + env.get("LD_LIBRARY_PATH", "")

# subprocess.run(
#     [str(VENV / "bin/python"), "-m", "bitsandbytes"],
#     env=env,
# )

In [7]:
# import json
# from pathlib import Path

# kernel_json = Path("/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json")

# with open(kernel_json, "r") as f:
#     spec = json.load(f)

# ld_library_path = (
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:"
#     "${LD_LIBRARY_PATH}"
# )

# spec.setdefault("env", {})
# spec["env"]["LD_LIBRARY_PATH"] = ld_library_path
# spec["env"]["BNB_CUDA_VERSION"] = "118"

# with open(kernel_json, "w") as f:
#     json.dump(spec, f, indent=2)

# print(kernel_json)
# print(json.dumps(spec, indent=2))

In [8]:
import transformers

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.3.1+cu121
cuda: 12.1
cuda available: True
transformers: 4.51.3


In [9]:
try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

bitsandbytes: 0.45.5


In [10]:
from transformers.utils import is_torch_available, is_bitsandbytes_available

print("is_torch_available:", is_torch_available())
print("is_bitsandbytes_available:", is_bitsandbytes_available())

is_torch_available: True
is_bitsandbytes_available: True


In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

from baseline.datasets import load_public_splits, load_private_set
from baseline.generation import GenerationConfig
from baseline.prompt_sets import build_prompt_texts
from baseline.modeling import ModelConfig, load_transformers_model, predownload_model
from baseline.scoring import load_judger, score_one, summarize_results
from baseline.progress_viz import RunProgressDashboard
from prompting.prompt_chain import build_prompt_chain
from baseline.runner import run_problem_set

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [12]:
splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=VAL_FRAC, seed=SPLIT_SEED)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

print("Train summary:")
pprint(train_set.summary())

print("\nValidation summary:")
pprint(val_set.summary())

print("\nPublic summary:")
pprint(public_set.summary())

print("\nPrivate summary:")
pprint(private_set.summary())

Train summary:
{'n': 901,
 'n_answered': 901,
 'n_free_form': 601,
 'n_mcq': 300,
 'name': 'public_train'}

Validation summary:
{'n': 225,
 'n_answered': 225,
 'n_free_form': 150,
 'n_mcq': 75,
 'name': 'public_val'}

Public summary:
{'n': 1126,
 'n_answered': 1126,
 'n_free_form': 751,
 'n_mcq': 375,
 'name': 'public'}

Private summary:
{'n': 943, 'n_answered': 0, 'n_free_form': 643, 'n_mcq': 300, 'name': 'private'}


In [13]:
prompt_chain = build_prompt_chain(strategy_name="baseline")

for label, problem_set in [("train", train_set), ("val", val_set), ("private", private_set)]:
    problem = problem_set.problems()[0]
    spec = prompt_chain.build_spec(problem)

    print("=" * 80)
    print(label, "id=", problem.id, "template=", spec.name)
    print("metadata:", spec.metadata)
    print("generation_hints:", spec.generation_hints)
    print(spec.to_messages()[0]["content"][:300])
    print("--- user ---")
    print(spec.to_messages()[-1]["content"][:500])

train id= 499 template= baseline_mcq
metadata: {'strategy_name': 'baseline', 'route_name': 'mcq', 'tags': []}
generation_hints: {'temperature': 0.6, 'top_p': 0.95}
You are an expert mathematician. Read the problem and the answer choices below, then select the single best answer. Output only the letter of your chosen option inside \boxed{}, e.g. \boxed{C}.
--- user ---
We now define an algorithm: The definition of a(n) is the least odd number k such that k * 2^n + 1 is a prime number. Given the input x_list (a series of values): [70, 71, 72, 73, 74, 75, 76, 77, 78, 79], determine the corresponding output sequence y_list.

Answer choices:
A. [44, 43, 129, 26, 63, 1, 90, 33, 22, 243]
B. [37, 35, 122, 19, 64, 10, 96, 26, 20, 245]
C. [38, 40, 128, 22, 71, 3, 91, 28, 14, 248]
D. [43, 37, 125, 21, 70, 9, 98, 27, 13, 246]
E. [39, 39, 127, 23, 67, 5, 93, 29, 15, 249]

val id= 990 template= baseline_free_form
metadata: {'strategy_name': 'baseline', 'route_name': 'free_form', 'tags': []}
generati

## 4. Modeling

In [14]:
model_config = ModelConfig(
    model_id=MODEL_ID,
    cache_dir=CACHE_DIR,
    gpu_id=GPU_ID,
    load_in_4bit=LOAD_IN_4BIT,
    torch_dtype="float16",
    max_input_tokens=MAX_INPUT_TOKENS,
    reuse_loaded=True,
)

if "model_bundle" in globals() and model_bundle.config.cache_key() == model_config.cache_key():
    print("Reusing notebook-level model_bundle.")
else:
    t0 = time.perf_counter()
    model_bundle = load_transformers_model(model_config)
    print(f"Model load/reuse time: {time.perf_counter() - t0:.2f} sec")

print("Model device:", model_bundle.device())

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

CUDA available: NVIDIA GeForce GTX 1080 Ti
Model load/reuse time: 10.03 sec
Model device: cuda:0


In [15]:
class NotebookGenerationConfig:
    def __init__(
        self,
        max_new_tokens=1024,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        do_sample=True,
    ):
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k
        self.repetition_penalty = repetition_penalty
        self.do_sample = do_sample


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_submission_csv(scored_rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "response"])
        writer.writeheader()
        for row in scored_rows:
            writer.writerow({
                "id": row["id"],
                "response": row["response"],
            })


def generate_batch(model_bundle, prompt_texts, generation_config):
    import torch

    tokenizer = model_bundle.tokenizer
    model = model_bundle.model
    device = model_bundle.device()

    inputs = tokenizer(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=model_bundle.config.max_input_tokens,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=generation_config.max_new_tokens,
            temperature=generation_config.temperature,
            top_p=generation_config.top_p,
            top_k=generation_config.top_k,
            repetition_penalty=generation_config.repetition_penalty,
            do_sample=generation_config.do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    responses = []

    for out in output_ids:
        new_tokens = out[prompt_len:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        responses.append(response)

    return responses


def run_problem_set_notebook(
    problem_set,
    run_name,
    model_bundle,
    prompt_chain,
    generation_config,
    batch_size=1,
    limit=None,
    score=True,
    output_jsonl_path=None,
    submission_csv_path=None,
    render_every=1,
):
    if limit is not None:
        problem_set = problem_set.head(limit)

    score_available = score and any("answer" in r and r.get("answer") is not None for r in problem_set.records)

    dashboard = RunProgressDashboard(
        run_name=run_name,
        total=len(problem_set),
        score_available=score_available,
        render_every=render_every,
        show_accuracy=score_available,
        show_timing=True,
    )

    t_prompt = time.perf_counter()
    prompt_rows = build_prompt_texts(problem_set, model_bundle.tokenizer, prompt_chain=prompt_chain)
    prompt_build_sec = time.perf_counter() - t_prompt

    judger = load_judger(".") if score_available else None

    all_rows = []
    total_generation_sec = 0.0

    for start_idx in range(0, len(prompt_rows), batch_size):
        batch_rows = prompt_rows[start_idx:start_idx + batch_size]
        batch_prompt_texts = [r["prompt_text"] for r in batch_rows]

        t_gen = time.perf_counter()
        responses = generate_batch(model_bundle, batch_prompt_texts, generation_config)
        batch_generation_sec = time.perf_counter() - t_gen
        total_generation_sec += batch_generation_sec

        for offset, (prompt_row, response) in enumerate(zip(batch_rows, responses)):
            record = problem_set.records[start_idx + offset]
            correct = score_one(record, response, judger=judger) if score_available else None

            result_row = {
                "id": record.get("id"),
                "is_mcq": bool(record.get("options")),
                "gold": record.get("answer"),
                "response": response,
                "correct": correct,
                "template_name": prompt_row["spec"].name,
                "prompt_metadata": prompt_row["metadata"],
            }

            all_rows.append(result_row)
            dashboard.update({
                "id": result_row["id"],
                "is_mcq": result_row["is_mcq"],
                "correct": result_row["correct"],
            })

    dashboard.finish()

    summary = summarize_results(all_rows)
    timing_summary = {
        "prompt_build_sec": prompt_build_sec,
        "generation_sec": total_generation_sec,
        "sec_per_problem_generation_only": total_generation_sec / len(all_rows) if all_rows else None,
    }

    if output_jsonl_path:
        save_jsonl(all_rows, output_jsonl_path)

    if submission_csv_path:
        save_submission_csv(all_rows, submission_csv_path)

    run_report = {
        "run_name": run_name,
        "problem_set": problem_set.summary(),
        "score_available": score_available,
        "summary": summary,
        "timings": timing_summary,
        "output_jsonl_path": str(output_jsonl_path) if output_jsonl_path else None,
        "submission_csv_path": str(submission_csv_path) if submission_csv_path else None,
    }

    return {
        "problem_set": problem_set,
        "prompt_rows": prompt_rows,
        "results": all_rows,
        "summary": summary,
        "timings": timing_summary,
        "dashboard": dashboard,
        "report": run_report,
    }

# Baseline 1: Naive

In [16]:
baseline1_generation_config = NotebookGenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_BASELINE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    repetition_penalty=1.0,
    do_sample=True,
)

baseline1_prompt_chain = build_prompt_chain(strategy_name="baseline")

TRAIN_LIMIT = 5
VAL_LIMIT = 5
PRIVATE_LIMIT = None

print("Baseline 1 config:")
print("max_new_tokens:", baseline1_generation_config.max_new_tokens)
print("batch_size:", BATCH_SIZE)
print("train_limit:", TRAIN_LIMIT)
print("val_limit:", VAL_LIMIT)
print("private_limit:", PRIVATE_LIMIT)

Baseline 1 config:
max_new_tokens: 1024
batch_size: 1
train_limit: 5
val_limit: 5
private_limit: None


In [17]:
# Train split smoke run

TRAIN_LIMIT = 5

baseline1_train_result = run_problem_set_notebook(
    problem_set=train_set,
    run_name="Baseline 1 - train split smoke run",
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=TRAIN_LIMIT,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "train_results.jsonl",
    render_every=1,
)

pprint(baseline1_train_result["report"])


KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
for row in baseline1_train_result["rows"]:
    print("=" * 80)
    print("id:", row["id"], "correct:", row["correct"])
    print(row["response"][:2000])

In [ ]:
# Validation run

VAL_LIMIT = 5 # None

baseline1_val_result = run_problem_set_notebook(
    problem_set=val_set,
    run_name="Baseline 1 - validation run",
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=VAL_LIMIT,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "val_results.jsonl",
    render_every=1,
)

pprint(baseline1_val_result["report"])

In [ ]:
# Test run

PRIVATE_LIMIT = None

baseline1_private_result = run_problem_set_notebook(
    problem_set=private_set,
    run_name="Baseline 1 - private test run",
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=PRIVATE_LIMIT,
    score=False,
    output_jsonl_path=BASELINE1_DIR / "private_results.jsonl",
    submission_csv_path=BASELINE1_DIR / "submission.csv",
    render_every=1,
)

pprint(baseline1_private_result["report"])

In [ ]:
submission_path = BASELINE1_DIR / "submission.csv"

if submission_path.exists():
    sub_df = pd.read_csv(submission_path)
    print(sub_df.shape)
    display(sub_df.head())
    print("Columns:", list(sub_df.columns))
else:
    print("No submission file yet. Run the private test cell first.")

# Baseline 2: Output hardening

# Baseline 3: Prompt Engineering

# Baseline 4: Supervised Fine-Tuning (SFT)

## Post-SFT Tuning

# Baseline 5: RL